# CNN-EXP-007 — BPE++ GPU

Меняем только выход модели: общий backbone из CNN-EXP-001 получает две головы.
Presence предсказывает `count > 0`, intensity будет предсказывать `log1p(count)` на положительных позициях.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import torch
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)

In [ ]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))

In [ ]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(split.intervals("train")["contig"])
validation_contigs = set(split.intervals("validation")["contig"])

assert train_contigs.isdisjoint(validation_contigs)

display(split.summary)
print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))

In [ ]:
profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(batch_size=4)

print("Обычный вход:", train_batch.x.shape)
print("Обе ориентации:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)
print("Mask:", train_batch.mask.shape)
print("Положительных:", int(train_batch.target.sum()))
print("Sampling branches:", train_batch.sampling_branch)

## Двухголовая модель

Backbone полностью совпадает с CNN-EXP-001. Добавляется только второй выходной канал.

In [ ]:
multitask_model = create_cnn_presence_intensity().to(device)
multitask_model.eval()

x_both_gpu = train_batch.x_both_strands.to(device)

with torch.no_grad():
    presence_logits, intensity_prediction = multitask_model(x_both_gpu)

parameter_count = sum(
    parameter.numel()
    for parameter in multitask_model.parameters()
    if parameter.requires_grad
)

expected_shape = (
    2 * len(train_batch.tiles),
    1,
    profile_config.input_length,
)

print("Общий вход:", x_both_gpu.shape, x_both_gpu.device)
print("Presence:", presence_logits.shape, presence_logits.device)
print("Intensity:", intensity_prediction.shape, intensity_prediction.device)
print("Параметров:", parameter_count)

assert presence_logits.shape == expected_shape
assert intensity_prediction.shape == expected_shape
assert parameter_count == 59138

In [ ]:
multitask_model.eval()

with torch.no_grad():
    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

print("Plus presence:", plus_presence.shape)
print("Minus presence reverse:", minus_presence_reverse.shape)
print("Plus intensity:", plus_intensity.shape)
print("Minus intensity reverse:", minus_intensity_reverse.shape)

assert plus_presence.shape == (4, 1, 1024)
assert minus_presence_reverse.shape == (4, 1, 1024)
assert plus_intensity.shape == (4, 1, 1024)
assert minus_intensity_reverse.shape == (4, 1, 1024)

In [ ]:
plus_count = (
    train_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    train_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target = torch.log1p(
    plus_count
)

minus_intensity_target_reverse = torch.log1p(
    minus_count_reverse
)

plus_intensity_mask = (
    plus_count > 0
)

minus_intensity_mask_reverse = (
    minus_count_reverse > 0
)

print("Plus count:", plus_count.shape)
print("Minus count reverse:", minus_count_reverse.shape)

print(
    "Положительных plus:",
    int(plus_intensity_mask.sum()),
)

print(
    "Положительных minus:",
    int(minus_intensity_mask_reverse.sum()),
)

print(
    "Диапазон intensity target:",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).min()
    ),
    "—",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).max()
    ),
)

assert torch.equal(
    plus_intensity_mask.float(),
    train_batch.plus_target,
)

assert torch.equal(
    minus_intensity_mask_reverse.float(),
    train_batch.minus_target_reverse,
)

In [ ]:
import warnings


INTENSITY_LOSS_WEIGHT = 0.1

multitask_model.train()

optimizer = DirectMLAdam(
    multitask_model.parameters(),
    lr=1e-3,
)

# Presence targets и веса.
plus_target_gpu = (
    train_batch.plus_target.to(device)
)
minus_target_reverse_gpu = (
    train_batch.minus_target_reverse.to(device)
)

plus_presence_weight_gpu = (
    train_batch.plus_loss_weight.to(device)
)
minus_presence_weight_reverse_gpu = (
    train_batch.minus_loss_weight_reverse.to(device)
)

# Intensity targets.
plus_intensity_target_gpu = (
    plus_intensity_target.to(device)
)
minus_intensity_target_reverse_gpu = (
    minus_intensity_target_reverse.to(device)
)

# Положительные веса в BCE нормированы на 0.5.
# Умножение на 2 превращает их в условное распределение
# только среди положительных position-strand.
plus_intensity_weight_gpu = (
    2.0
    * train_batch.plus_loss_weight
    * plus_intensity_mask.float()
).to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * train_batch.minus_loss_weight_reverse
    * minus_intensity_mask_reverse.float()
).to(device)

optimizer.zero_grad(set_to_none=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        + dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT * intensity_loss
    )

    total_loss.backward()
    optimizer.step()

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

presence_gradient = (
    multitask_model.profile_head.weight.grad[0]
    .abs()
    .sum()
    .item()
)

intensity_gradient = (
    multitask_model.profile_head.weight.grad[1]
    .abs()
    .sum()
    .item()
)

print("Presence loss:", presence_loss.item())
print("Intensity loss:", intensity_loss.item())
print("Total loss:", total_loss.item())
print("Presence gradient:", presence_gradient)
print("Intensity gradient:", intensity_gradient)
print("CPU fallback:", fallback_warnings)

assert presence_gradient > 0
assert intensity_gradient > 0
assert not fallback_warnings

In [ ]:
import math
import time
import warnings


PILOT_STEPS = 200
PILOT_BATCH_SIZE = 4
PILOT_SEED = 42
LOG_EVERY = 20
INTENSITY_LOSS_WEIGHT = 0.1

torch.manual_seed(PILOT_SEED)

pilot_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=PILOT_SEED,
)

pilot_model = (
    create_cnn_presence_intensity()
    .to(device)
)

pilot_optimizer = DirectMLAdam(
    pilot_model.parameters(),
    lr=1e-3,
)

pilot_model.train()

pilot_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, PILOT_STEPS + 1):
        batch = pilot_generator.sample_batch(
            batch_size=PILOT_BATCH_SIZE
        )

        # Вход обеих ориентаций уже объединён на CPU.
        x_gpu = batch.x_both_strands.to(device)

        # Presence targets.
        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Intensity targets готовим и разворачиваем на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        pilot_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            pilot_model,
            x_gpu,
            batch_size=PILOT_BATCH_SIZE,
            context_flank=profile_config.context_flank,
            target_length=profile_config.target_length,
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT * intensity_loss
        )

        if not math.isfinite(total_loss.item()):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        pilot_optimizer.step()

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.95 * ema_presence
                + 0.05 * presence_value
            )
            ema_intensity = (
                0.95 * ema_intensity
                + 0.05 * intensity_value
            )
            ema_total = (
                0.95 * ema_total
                + 0.05 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        pilot_history.append(
            {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
            }
        )

        if step == 1 or step % LOG_EVERY == 0:
            elapsed = (
                time.perf_counter()
                - started_at
            )

            print(
                f"step={step:3d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

print()
print("Время:", round(elapsed, 2), "сек")
print("Финальный EMA presence:", round(ema_presence, 6))
print("Финальный EMA intensity:", round(ema_intensity, 6))
print("Финальный EMA total:", round(ema_total, 6))
print("CPU fallback:", fallback_warnings)

assert not fallback_warnings

In [ ]:
from pathlib import Path


EXP007_SEED = 42
EXP007_STEPS = 10_0000
EXP007_BATCH_SIZE = 4
EXP007_LOG_EVERY = 250
EXP007_CHECKPOINT_STEPS = {
    2_000,
    5_000,
    10_000,
}
INTENSITY_LOSS_WEIGHT = 0.1

EXP007_DIR = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP007"
    / "run_001"
)

EXP007_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.manual_seed(EXP007_SEED)

exp002_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=EXP007_SEED,
)

exp002_model = (
    create_cnn_presence_intensity()
    .to(device)
)

exp002_optimizer = DirectMLAdam(
    exp002_model.parameters(),
    lr=1e-3,
)

print("Артефакты:", EXP007_DIR)
print("Параметров:", sum(
    parameter.numel()
    for parameter in exp002_model.parameters()
    if parameter.requires_grad
))
print("Шагов:", EXP007_STEPS)
print("Intensity weight:", INTENSITY_LOSS_WEIGHT)

In [ ]:
import copy
import math
import time
import warnings

import pandas as pd


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def save_exp002_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": "CNN-EXP-007",
        "model_type": "presence_intensity",
        "step": step,
        "model_state_dict": model_state_on_cpu(
            exp002_model
        ),
        "optimizer_state_dict": tree_to_cpu(
            exp002_optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            exp002_generator.rng.bit_generator.state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "profile_config": {
            "target_length": (
                profile_config.target_length
            ),
            "context_flank": (
                profile_config.context_flank
            ),
            "positive_branch_fraction": (
                profile_config
                .positive_branch_fraction
            ),
        },
        "training_config": {
            "seed": EXP007_SEED,
            "steps": EXP007_STEPS,
            "batch_size": EXP007_BATCH_SIZE,
            "learning_rate": 1e-3,
            "intensity_loss_weight": (
                INTENSITY_LOSS_WEIGHT
            ),
            "huber_delta": 1.0,
        },
        "seen_position_strand": (
            seen_position_strand
        ),
        "elapsed_seconds": elapsed_seconds,
        "history": list(history),
    }

    torch.save(checkpoint, path)


exp002_model.train()

exp002_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, EXP007_STEPS + 1):
        batch = exp002_generator.sample_batch(
            batch_size=EXP007_BATCH_SIZE
        )

        x_gpu = batch.x_both_strands.to(device)

        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Всё выравнивание minus выполняется на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        exp002_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            exp002_model,
            x_gpu,
            batch_size=EXP007_BATCH_SIZE,
            context_flank=(
                profile_config.context_flank
            ),
            target_length=(
                profile_config.target_length
            ),
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        exp002_optimizer.step()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )
            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )
            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        if (
            step == 1
            or step % EXP007_LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - started_at
            )

            record = {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
                "elapsed_seconds": elapsed,
                "seen_position_strand": (
                    seen_position_strand
                ),
                "positions_per_second": (
                    seen_position_strand / elapsed
                ),
            }

            exp002_history.append(record)

            print(
                f"step={step:5d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

        if step in EXP007_CHECKPOINT_STEPS:
            checkpoint_path = (
                EXP007_DIR
                / f"model_step_{step:05d}.pt"
            )

            save_exp002_checkpoint(
                checkpoint_path,
                step=step,
                elapsed_seconds=(
                    time.perf_counter()
                    - started_at
                ),
                seen_position_strand=(
                    seen_position_strand
                ),
                history=exp002_history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

final_checkpoint_path = (
    EXP007_DIR / "exp002_final.pt"
)
history_path = (
    EXP007_DIR / "training_history.csv"
)

save_exp002_checkpoint(
    final_checkpoint_path,
    step=EXP007_STEPS,
    elapsed_seconds=elapsed,
    seen_position_strand=seen_position_strand,
    history=exp002_history,
)

pd.DataFrame(exp002_history).to_csv(
    history_path,
    index=False,
)

print()
print("Общее время:", round(elapsed, 2), "сек")
print(
    "Время шага:",
    round(elapsed / EXP007_STEPS, 4),
    "сек",
)
print(
    "Увидено position-strand:",
    seen_position_strand,
)
print(
    "Финальный EMA presence:",
    round(ema_presence, 6),
)
print(
    "Финальный EMA intensity:",
    round(ema_intensity, 6),
)
print(
    "Финальный EMA total:",
    round(ema_total, 6),
)
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", final_checkpoint_path)
print("История:", history_path)

assert not fallback_warnings
assert final_checkpoint_path.is_file()
assert history_path.is_file()

In [ ]:
from ml_project.epic_cnn.evaluation import (
    evaluate_presence_intensity_model,
    load_cnn_presence_intensity_checkpoint,
)


EXP007_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP007"
    / "run_001"
    / "exp002_final.pt"
)

loaded_exp002 = (
    load_cnn_presence_intensity_checkpoint(
        EXP007_CHECKPOINT_PATH,
        device=device,
    )
)

validation_model = loaded_exp002.model
validation_config = loaded_exp002.profile_config
checkpoint = loaded_exp002.checkpoint

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=validation_config,
    seed=42,
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_generator.tile_index))
print(
    "Validation position-strand:",
    int(
        validation_generator
        .tile_index["allowed_position_strand"]
        .sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)
print(
    "Device:",
    next(validation_model.parameters()).device,
)

In [ ]:
validation_result = (
    evaluate_presence_intensity_model(
        validation_model,
        validation_generator,
        device=device,
        batch_size=8,
        score_decimals=5,
        progress_every_batches=500,
    )
)

In [ ]:
VALIDATION_METRICS_PATH = (
    EXP007_CHECKPOINT_PATH.parent
    / "validation_metrics.csv"
)

validation_result.metrics.to_csv(
    VALIDATION_METRICS_PATH,
    index=False,
)

overall_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        == "overall"
    ]
    .reset_index(drop=True)
)

display(overall_metrics)

print(
    "Validation time:",
    round(validation_result.elapsed_seconds, 2),
    "sec",
)
print(
    "Position-strand/s:",
    f"{validation_result.positions_per_second:,.0f}",
)
print(
    "CPU fallback:",
    list(validation_result.fallback_warnings),
)
print("Metrics:", VALIDATION_METRICS_PATH)

In [ ]:
per_contig_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        != "overall"
    ]
    .sort_values(
        ["segment", "score"]
    )
    .reset_index(drop=True)
)

display(per_contig_metrics)

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "RF++ охват очень сильно влияет нужно смотреть шире!"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-007',
        title='RF++',
        hypothesis='увеличиваю ещё сильнее RF до максимума',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP007_rf.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP007/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-003',
        reference_metrics='artifacts/experiments/CNN_EXP003/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "увеличенное число шагов сильно улучшает AP"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-007',
        title='STEP++ exp',
        hypothesis='увеличение шагов даст  явный прирост AP',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP007_step_exp.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP007/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-004',
        reference_metrics='artifacts/experiments/CNN_EXP004/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = False
FINAL_DECISION = "review"  # review / adopt / reject / iterate
FINAL_NOTES = ""

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-007',
        title='BPE++ GPU',
        hypothesis='Увеличить BPE мерджи и перенести вычисления на GPU',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP007_bpe_gpu.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP007/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-005',
        reference_metrics='artifacts/experiments/CNN_EXP005/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")